In [ ]:
import os
from cryptography.hazmat.primitives import padding
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes

#### Padding

In [ ]:
BLOCK_SIZE = 128

def pad(ptxt):
    padder = padding.PKCS7(BLOCK_SIZE).padder()
    padded = padder.update(ptxt) + padder.finalize()
    return padded

def ints(bstr):
    return [int(x) for x in bstr]

ints(pad(b"abc"))

In [ ]:
def unpad(padded):
    unpadder = padding.PKCS7(BLOCK_SIZE).unpadder()
    ptxt = unpadder.update(padded) + unpadder.finalize()
    return ptxt

# removendo um bytes...
unpad(pad(b"abc"))

#### Questão 01
Ilustre (com um fragmento de código) um erro no unpadding que não se deva ao facto da mensagem não ser múltipla do tamanho do bloco.

In [ ]:
mensagem_padded = pad(b"abc") 
# transformar em 'bytearray' para podermos modificar um valor
mensagem_corrompida = bytearray(mensagem_padded)

# Alteramos o último byte. 
# O PKCS7 espera que o último byte indique o tamanho do padding.
# Se o padding original era 13, o último byte é 13 (0x0d). 
# Vamos mudá-lo para um valor arbitrário, como 50.
mensagem_corrompida[-1] = 50 
print(f"Tamanho da mensagem: {len(mensagem_corrompida)} bytes") # Continua múltiplo de 16
# Tentar fazer o unpad causará o erro de formato
try:
    unpad(bytes(mensagem_corrompida))
except Exception as e:
    print(f"Erro capturado: {e}")

#### Padding oracle

In [ ]:
def pad_orcl(ctxt):
    padded_msg = dec(ctxt) # função que decifra o criptograma
    unpadder = padding.PKCS7(BLOCK_SIZE * 8).unpadder()
    res = True
    try:
        ptxt = unpadder.update(padded_msg) + unpadder.finalize()
    except ValueError:
        res = False
    return res # retorna apenas se houve ou não erro no "unpadding"

#### PROG: cbc_pad_orcl

In [ ]:
BLOCK_SIZE = 16
KEY = os.urandom(32)
IV = os.urandom(16)

def encrypt(plaintext):
    """Cifra a mensagem usando AES-CBC com Padding PKCS7."""
    padder = padding.PKCS7(BLOCK_SIZE * 8).padder()
    padded_data = padder.update(plaintext.encode()) + padder.finalize()
    cipher = Cipher(algorithms.AES(KEY), modes.CBC(IV))
    encryptor = cipher.encryptor()
    return encryptor.update(padded_data) + encryptor.finalize()

def pad_orcl(ctxt):
    """
    O Oráculo de Padding: 
    Retorna True se o padding estiver correto, False caso contrário.
    """
    try:
        cipher = Cipher(algorithms.AES(KEY), modes.CBC(IV))
        decryptor = cipher.decryptor()
        decrypted_msg = decryptor.update(ctxt) + decryptor.finalize()
        
        unpadder = padding.PKCS7(BLOCK_SIZE * 8).unpadder()
        unpadder.update(decrypted_msg)
        unpadder.finalize()
        
        return True
    except ValueError:
        return False

if __name__ == "__main__":
    msg = "Segurança Informática é brutal!"
    
    criptograma_valido = encrypt(msg)
    
    lista_bytes = bytearray(criptograma_valido)
    lista_bytes[-1] ^= 0xFF # Corromper o último byte
    criptograma_invalido = bytes(lista_bytes)
    
    print(f"Teste 1 (Válido): {pad_orcl(criptograma_valido)}")
    print(f"Teste 2 (Inválido): {pad_orcl(criptograma_invalido)}")

### Padding oracle attack

##### PROG: pad_orcl_attck_lastbyte


In [ ]:
def padding_size(ctxt):
    """
    Determina o tamanho do padding modificando o penúltimo bloco.
    """
    full_ctxt = bytearray(IV + ctxt)
    
    last_block_start = len(full_ctxt) - BLOCK_SIZE
    prev_block_start = last_block_start - BLOCK_SIZE

    print(f"A analisar criptograma de {len(full_ctxt)} bytes...")

    # Vamos alterar os bytes do bloco anterior um a um, da esquerda para a direita
    for i in range(BLOCK_SIZE):
        # Guardamos o byte original para restaurar depois
        original_byte = full_ctxt[prev_block_start + i]
        
        # Alteramos o byte (fazendo XOR com 1, dica do guião)
        full_ctxt[prev_block_start + i] ^= 0x01
        
        test_ctxt = bytes(full_ctxt[BLOCK_SIZE:])
        
        if not pad_orcl(test_ctxt):
            padding_size = BLOCK_SIZE - i
            return padding_size
        
        full_ctxt[prev_block_start + i] = original_byte

    return 0

if __name__ == "__main__":
    msg = "Olá" 
    criptograma = encrypt(msg)
    tamanho_encontrado = padding_size(criptograma)
    
    print(f"\nResultado:")
    print(f"Tamanho do padding detetado pelo ataque: {tamanho_encontrado} bytes")
    print(f"Bytes de mensagem real: {len(criptograma) - tamanho_encontrado}")

##### PROG: pad_orcl_attck


In [ ]:
def attack_block(target_block, prev_block):
    """Recupera o texto limpo de um bloco específico."""
    decrypted_intermediate = bytearray(BLOCK_SIZE)
    plaintext_block = bytearray(BLOCK_SIZE)
    
    # Atacar de trás para a frente (do byte 15 ao 0)
    for i in range(1, BLOCK_SIZE + 1):
        padding_value = i
        # Bloco de Criptograma Falso (C'n-1)
        fake_iv = bytearray(BLOCK_SIZE)
        for j in range(1, i):
            fake_iv[-j] = decrypted_intermediate[-j] ^ padding_value
            
        found = False
        for val in range(256):
            fake_iv[-i] = val
            if pad_orcl(target_block, bytes(fake_iv)):
                decrypted_intermediate[-i] = val ^ padding_value
                found = True
                break
        
        if not found:
            raise Exception(f"Não foi possível encontrar o byte no índice {-i}")

    for k in range(BLOCK_SIZE):
        plaintext_block[k] = decrypted_intermediate[k] ^ prev_block[k]
        
    return plaintext_block

if __name__ == "__main__":
    msg_secreta = "Mensagem Secreta"
    ctxt = encrypt(msg_secreta)
    
    bloco_alvo = ctxt[0:16]
    bloco_anterior = IV
    
    print("A iniciar ataque ao bloco...")
    bloco_decifrado = attack_block(bloco_alvo, bloco_anterior)
    
    print(f"\n[Sucesso!]")
    print(f"Texto limpo recuperado: {bloco_decifrado}")
    print(f"Texto em string: {bloco_decifrado.decode('ascii', errors='ignore')}")